# EDA Consolidado - Panvel

Este notebook consolida o EDA feito pelo grupo em uma narrativa unica. Os notebooks individuais continuam em `notebooks/00_referencias/`, mas este arquivo passa a ser a versao oficial para revisao e apresentacao.

Fontes usadas:

- Arthur: qualidade de metas, vendas sem meta, calendario e datas comerciais.
- Gustavo: perfil de filiais, ticket medio, agregacoes temporais e ideias de features de modelagem.
- Celso: preparacao das bases, `Base_V1`, `Base_exo`, `Base_V2` e clusterizacao.
- BC: relatorios de EDA, devolucoes, performance mensal e documentacao estruturada.


## Objetivos

1. Conferir a qualidade e consistencia das bases.
2. Entender o escopo de filiais usado na pipeline.
3. Consolidar achados sobre vendas, metas, devolucoes e filiais.
4. Registrar decisoes para preparacao, features e modelagem.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

ROOT = Path.cwd()
BASE_ORIGI = ROOT / "Base_Origi"
BASE_V1 = ROOT / "Base_V1"
BASE_EXO = ROOT / "Base_exo"
BASE_V2 = ROOT / "Base_V2"

ARQUIVOS = {
    "orig_vendas": BASE_ORIGI / "project-puc_vendas.parquet",
    "orig_metas": BASE_ORIGI / "project-puc_metas.parquet",
    "orig_filiais": BASE_ORIGI / "project-puc_filiais.parquet",
    "orig_filiais_dt": BASE_ORIGI / "project-puc_filiais_dt_abertura.parquet",
    "orig_devolucoes": BASE_ORIGI / "project-puc_devolucoes.parquet",
    "v1_vendas": BASE_V1 / "vendas_V1.parquet",
    "v1_metas": BASE_V1 / "metas_V1.parquet",
    "v1_filiais": BASE_V1 / "filiais_V1.parquet",
    "v1_vendas_diaria": BASE_V1 / "vendas_diaria_V1.parquet",
    "exo_vendas": BASE_EXO / "vendas_exo.parquet",
    "exo_metas": BASE_EXO / "metas_exo.parquet",
    "exo_filiais": BASE_EXO / "filiais_exo.parquet",
    "exo_vendas_diaria": BASE_EXO / "vendas_diaria_exo.parquet",
    "v2_diarias": BASE_V2 / "features_filiais_diarias_V2.parquet",
    "v2_agregadas": BASE_V2 / "features_filiais_agregadas_V2.parquet",
    "v2_cluster_matrix": BASE_V2 / "features_filiais_cluster_semana_dia_V2.parquet",
    "v2_cluster": BASE_V2 / "cluster_filial_modelagem_V2.parquet",
    "v2_perfil": BASE_V2 / "perfil_clusters_V2.parquet",
}

def parquet_info(path: Path) -> dict:
    pf = pq.ParquetFile(path)
    return {
        "linhas": pf.metadata.num_rows,
        "colunas": len(pf.schema_arrow.names),
        "tamanho_mb": path.stat().st_size / 1024**2,
    }

def read_cols(nome: str, cols: list[str]) -> pd.DataFrame:
    return pd.read_parquet(ARQUIVOS[nome], columns=cols)


## 1. Inventario das bases

Primeiro conferimos tamanho, linhas e colunas de cada camada usada no projeto.

In [ ]:
inventario = (
    pd.DataFrame.from_dict({nome: parquet_info(path) for nome, path in ARQUIVOS.items()}, orient="index")
    .reset_index(names="base")
)
inventario["linhas"] = inventario["linhas"].astype(int)
inventario["colunas"] = inventario["colunas"].astype(int)
inventario["tamanho_mb"] = inventario["tamanho_mb"].round(2)
inventario

## 2. Cobertura de filiais

Este bloco consolida a analise feita principalmente no EDA do Celso e nas notas do BC: existe uma filial transacional (`1704`) que nao aparece no cadastro, e a pipeline separa 100 filiais principais + 24 filiais `Base_exo`.

In [ ]:
filiais_por_base = {}

for nome, path in ARQUIVOS.items():
    cols = pq.ParquetFile(path).schema_arrow.names
    if "codigo_filial" in cols:
        s = pd.read_parquet(path, columns=["codigo_filial"])["codigo_filial"].astype(str)
        filiais_por_base[nome] = set(s.unique())

pd.DataFrame(
    [{"base": nome, "filiais_distintas": len(filiais)} for nome, filiais in filiais_por_base.items()]
).sort_values("base")

In [ ]:
orig = filiais_por_base["orig_filiais"]
v1 = filiais_por_base["v1_filiais"]
exo = filiais_por_base["exo_filiais"]
v2 = filiais_por_base["v2_diarias"]

checks_filiais = {
    "filiais_originais": len(orig),
    "filiais_v1": len(v1),
    "filiais_exo": len(exo),
    "v1_mais_exo": len(v1 | exo),
    "intersecao_v1_exo": len(v1 & exo),
    "orig_menos_v1_exo": sorted(orig - (v1 | exo)),
    "v1_exo_menos_orig": sorted((v1 | exo) - orig),
    "v2_igual_v1": v2 == v1,
}

checks_filiais

### Filial 1704

A filial `1704` aparece em vendas, metas e devolucoes, mas nao aparece em `filiais` nem em `filiais_dt_abertura`. A decisao atual e manter fora da modelagem.

In [ ]:
linhas_1704 = []
for nome, data_col in [
    ("orig_vendas", "data_emissao"),
    ("orig_metas", "data_meta_venda"),
    ("orig_devolucoes", "data_devolucao"),
    ("orig_filiais", None),
    ("orig_filiais_dt", None),
]:
    cols = ["codigo_filial"] + ([data_col] if data_col else [])
    df = read_cols(nome, cols)
    mask = df["codigo_filial"].astype(str).eq("1704")
    row = {"base": nome, "linhas_1704": int(mask.sum())}
    if data_col and mask.any():
        datas = pd.to_datetime(df.loc[mask, data_col], errors="coerce")
        row.update({"data_min": datas.min(), "data_max": datas.max(), "datas_distintas": datas.dt.date.nunique()})
    linhas_1704.append(row)

pd.DataFrame(linhas_1704)

## 3. Datas e calendario

As bases principais cobrem 2024 e 2025. A checagem tambem identifica pequenas ausencias que precisam ser consideradas antes da modelagem.

In [ ]:
date_specs = {
    "orig_vendas": "data_emissao",
    "orig_metas": "data_meta_venda",
    "orig_devolucoes": "data_devolucao",
    "v1_vendas": "data_emissao_data",
    "v1_metas": "data_meta_venda_data",
    "v1_vendas_diaria": "data",
    "v2_diarias": "data",
}

linhas_datas = []
for nome, col in date_specs.items():
    s = pd.to_datetime(read_cols(nome, [col])[col], errors="coerce")
    linhas_datas.append({
        "base": nome,
        "coluna": col,
        "data_min": s.min(),
        "data_max": s.max(),
        "nulos": int(s.isna().sum()),
        "datas_distintas": int(s.dt.date.nunique()),
    })

pd.DataFrame(linhas_datas)

In [ ]:
v2_datas = read_cols("v2_diarias", ["codigo_filial", "data"])
dias_por_filial_v2 = (
    v2_datas.assign(codigo_filial=lambda d: d["codigo_filial"].astype(str))
    .groupby("codigo_filial")["data"]
    .nunique()
    .sort_values()
)

dias_por_filial_v2[dias_por_filial_v2 < 731]

## 4. Nulos e duplicatas

Este bloco consolida a parte de qualidade dos dados. As bases oficiais checadas nao apresentaram nulos nem duplicatas nas chaves principais.

In [ ]:
bases_nulos = ["orig_filiais", "orig_filiais_dt", "orig_metas", "orig_devolucoes", "orig_vendas", "v1_filiais", "v1_metas", "v1_vendas_diaria", "v2_diarias", "v2_agregadas", "v2_cluster"]

resumo_nulos = []
for nome in bases_nulos:
    df = pd.read_parquet(ARQUIVOS[nome])
    nulls = df.isna().sum()
    resumo_nulos.append({
        "base": nome,
        "colunas_com_nulos": int((nulls > 0).sum()),
        "total_nulos": int(nulls.sum()),
    })

pd.DataFrame(resumo_nulos)

In [ ]:
checks_chaves = [
    ("orig_filiais", ["codigo_filial"]),
    ("orig_filiais_dt", ["codigo_filial"]),
    ("orig_metas", ["codigo_filial", "data_meta_venda"]),
    ("v1_filiais", ["codigo_filial"]),
    ("v1_metas", ["codigo_filial", "data_meta_venda_data"]),
    ("v1_vendas_diaria", ["codigo_filial", "data"]),
    ("exo_filiais", ["codigo_filial"]),
    ("exo_metas", ["codigo_filial", "data_meta_venda_data"]),
    ("exo_vendas_diaria", ["codigo_filial", "data"]),
    ("v2_agregadas", ["codigo_filial"]),
    ("v2_cluster", ["codigo_filial"]),
]

duplicatas = []
for nome, keys in checks_chaves:
    df = read_cols(nome, keys)
    duplicatas.append({"base": nome, "chave": " + ".join(keys), "linhas_duplicadas": int(df.duplicated(keys).sum())})

pd.DataFrame(duplicatas)

## 5. Vendas e faturamento

Consolidando os EDAs: todos tratam `faturamento` como faturamento bruto. O liquido e calculado apenas quando juntamos devolucoes, mas a recomendacao atual e usar o bruto como alvo principal.

In [ ]:
def resumo_vendas(nome: str) -> dict:
    df = read_cols(nome, ["categoria_gerencial", "faturamento", "quantidade"])
    return {
        "base": nome,
        "linhas": len(df),
        "faturamento_total": df["faturamento"].sum(),
        "quantidade_total": df["quantidade"].sum(),
        "faturamento_negativo": int((df["faturamento"] < 0).sum()),
        "quantidade_negativa": int((df["quantidade"] < 0).sum()),
        "categorias": df["categoria_gerencial"].value_counts(dropna=False).to_dict(),
    }

pd.DataFrame([resumo_vendas("orig_vendas"), resumo_vendas("v1_vendas")])

## 6. Metas

A analise do Arthur chamou atencao para metas zeradas e qualidade de metas. Na `Base_V1`, metas negativas foram tratadas, mas ainda existem metas zeradas.

In [ ]:
def resumo_metas(nome: str) -> dict:
    df = pd.read_parquet(ARQUIVOS[nome])
    return {
        "base": nome,
        "linhas": len(df),
        "zero_valor_meta": int((df["valor_meta_venda"] == 0).sum()),
        "neg_meta_med": int((df["meta_med"] < 0).sum()),
        "neg_meta_n_med": int((df["meta_n_med"] < 0).sum()),
        "neg_valor_meta": int((df["valor_meta_venda"] < 0).sum()),
    }

pd.DataFrame([resumo_metas("orig_metas"), resumo_metas("v1_metas")])

In [ ]:
metas_v1 = read_cols("v1_metas", ["codigo_filial", "data_meta_venda_data", "valor_meta_venda", "meta_med", "meta_n_med"])
metas_zero = metas_v1[metas_v1["valor_meta_venda"].eq(0)].copy()

display({
    "linhas_meta_zero": len(metas_zero),
    "filiais_com_meta_zero": metas_zero["codigo_filial"].astype(str).nunique(),
    "data_min": pd.to_datetime(metas_zero["data_meta_venda_data"]).min(),
    "data_max": pd.to_datetime(metas_zero["data_meta_venda_data"]).max(),
})

metas_zero["codigo_filial"].astype(str).value_counts().head(10)

## 7. Devolucoes

As devolucoes sao importantes para diagnostico, mas nao devem substituir o faturamento bruto como alvo sem validacao de regra de negocio. Ha dias em que devolucao supera faturamento bruto, gerando liquido negativo.

In [ ]:
vendas_diaria = read_cols("v1_vendas_diaria", ["codigo_filial", "data", "faturamento_bruto_dia", "valor_devolucao_dia", "faturamento_liquido_dia"])

neg_liquido = vendas_diaria[vendas_diaria["faturamento_liquido_dia"] < 0].copy()
neg_liquido.sort_values("faturamento_liquido_dia")

## 8. Perfil das filiais

Este bloco consolida o perfil trabalhado pelo grupo: idade/faixa de vida, localidade, tipo de estabelecimento, metragem e servicos da loja.

In [ ]:
filiais = pd.read_parquet(ARQUIVOS["orig_filiais"])

for col in ["faixa_vida", "localidade", "tipo_estabelecimento", "delivery", "panvel_clinic", "estacionamento", "atendimento_24_horas"]:
    print(f"\n## {col}")
    display(filiais[col].value_counts(dropna=False).head(12).to_frame("quantidade"))

filiais[["metragem_area_venda"]].describe()

## 9. Clusterizacao

A clusterizacao atual cobre as 100 filiais elegiveis da `Base_V1`. O resultado fica como contexto para modelagem ou segmentacao posterior.

In [ ]:
cluster = pd.read_parquet(ARQUIVOS["v2_cluster"])

display(cluster["cluster_filial_nome"].value_counts().to_frame("filiais"))
display(cluster["flag_outlier_cluster"].value_counts().to_frame("filiais"))
cluster.head()

## 10. Decisoes para as proximas etapas

Decisoes consolidadas:

1. `faturamento` deve ser tratado como faturamento bruto.
2. O alvo principal inicial da modelagem deve ser faturamento bruto.
3. Devolucoes entram como diagnostico ou feature auxiliar, nao como alvo principal.
4. A filial `1704` deve ficar fora da modelagem enquanto nao houver cadastro confiavel.
5. `Base_V1` + `Base_exo` cobre as 124 filiais cadastradas; `Base_V2` cobre as 100 filiais elegiveis.
6. Antes de modelar, decidir tratamento das metas zeradas e da meta ausente da filial `1746` em 2024-02-04.
7. Avaliar se as ausencias de 2024-01-01 em 18 filiais na `Base_V2` devem ser preenchidas ou mantidas.
